# Training a football detector (YOLO26 + SoccerNet)

I'm fine-tuning YOLO26 on the SoccerNet-Tracking dataset to detect players, goalkeepers,
referees, and the ball, so I can use it in my PitchVision app.

- **Model:** YOLO26n (the nano version, fast enough to run on CPU)
- **Dataset:** [SoccerNet-Tracking](https://github.com/SoccerNet/sn-tracking)
- **Classes:** `player`, `goalkeeper`, `referee`, `ball`

Takes a few hours on a Kaggle GPU. At the end you get a `best.pt` file to download — see
the last cell for where it goes.


## Before running this

1. Get a SoccerNet password by signing their NDA: https://github.com/SoccerNet/sn-tracking
2. Add it as a Kaggle Secret called `SOCCERNET_PASSWORD` (Add-ons -> Secrets). Don't paste the password directly into a cell.
3. Turn on **Internet** in the notebook settings (off by default) or the pip install below will fail. You may need a verified phone number on your Kaggle account to enable it, and if you turn it on mid-session you'll need to restart the session.
4. Turn on a **GPU**: Settings -> Accelerator -> T4 x2 or P100.

I didn't have access to the real (NDA-gated) files while writing this, so I added a couple
safety nets: Step 2 prints the actual folder structure before assuming anything about it,
and Step 5 shows one converted image before spending GPU hours training on the whole thing.


In [ ]:
# need a recent ultralytics version for YOLO26
!pip install -q "ultralytics>=8.4.60"
!pip install -q SoccerNet
!pip install -q opencv-python-headless pandas pyyaml tqdm matplotlib


In [ ]:
import configparser
import random
import re
import shutil
import zipfile
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import pandas as pd
import torch
import yaml
from tqdm import tqdm
from ultralytics import YOLO

random.seed(0)

# paths
WORK_DIR = Path("/kaggle/working")
SOCCERNET_DIR = WORK_DIR / "SoccerNet"               # raw download goes here
YOLO_DATASET_DIR = WORK_DIR / "pitchvision_dataset"  # converted dataset goes here
RUNS_DIR = WORK_DIR / "runs"

# classes I want to detect (same 4 the app already uses)
CLASSES = ["player", "goalkeeper", "referee", "ball"]
CLASS_TO_ID = {c: i for i, c in enumerate(CLASSES)}

# data prep settings
SPLITS_TO_DOWNLOAD = ["train"]  # only split with labels, test/challenge don't ship gt.txt
FRAME_STRIDE = 5                # keep every 5th frame, neighbouring frames look near-identical
VAL_FRACTION = 0.15             # split by sequence so train/val don't share near-duplicate frames

# training settings
BASE_MODEL = "yolo26n.pt"
IMGSZ = 960          # same size the app uses at inference time
EPOCHS = 100
BATCH = 16
PATIENCE = 20        # stop early if val score stalls for this many epochs
DEVICE = 0           # use [0, 1] if you turned on a dual-GPU accelerator
RUN_NAME = "pitchvision_yolo26_soccernet"

print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))


## Step 1 — Download the dataset

Downloads both years of SoccerNet tracking data (`tracking` and `tracking-2023`) using
the Kaggle secret from before, so the password never shows up in the notebook.

Each download gets its own cell with disk space printed before/after, so I can tell if
something stalls. If a cell hangs for a few minutes with nothing new printing, stop it
and try again rather than waiting forever.


In [ ]:
from kaggle_secrets import UserSecretsClient
from SoccerNet.Downloader import SoccerNetDownloader

user_secrets = UserSecretsClient()
soccernet_password = user_secrets.get_secret("SOCCERNET_PASSWORD")

SOCCERNET_DIR.mkdir(parents=True, exist_ok=True)
downloader = SoccerNetDownloader(LocalDirectory=str(SOCCERNET_DIR))
downloader.password = soccernet_password

def print_disk_usage(label: str):
    total, used, free = shutil.disk_usage(WORK_DIR)
    print(f"[{label}] disk: {used / 1e9:.1f} GB used, {free / 1e9:.1f} GB free "
          f"(of {total / 1e9:.1f} GB total)")

print_disk_usage("baseline, before any download")


In [ ]:
# task name might change in a future SoccerNet release, check their repo if this breaks
downloader.downloadDataTask(task="tracking", split=SPLITS_TO_DOWNLOAD)
print_disk_usage("after 'tracking'")


In [ ]:
downloader.downloadDataTask(task="tracking-2023", split=SPLITS_TO_DOWNLOAD)
print_disk_usage("after 'tracking-2023'")


In [ ]:
# extract any zip files (some tasks come zipped, some don't)
for zip_path in SOCCERNET_DIR.rglob("*.zip"):
    extract_dir = zip_path.with_suffix("")
    if not extract_dir.exists():
        print("Extracting", zip_path.relative_to(SOCCERNET_DIR))
        with zipfile.ZipFile(zip_path) as zf:
            zf.extractall(extract_dir)

print_disk_usage("after extraction")
print("Download complete.")


## Step 2 — Look at what we downloaded

SoccerNet-Tracking uses the MOT20 format:

- `gt/gt.txt` — boxes only, no class column: `frame, track_id, bb_left, bb_top, bb_width, bb_height, conf, x, y, z`
- `gameinfo.ini` — maps each `track_id` to a role (player/goalkeeper/referee/ball), e.g. `trackletID_1=player; team left`
- `seqinfo.ini` — frame rate, resolution, image folder name
- `img1/000001.jpg`, ... — the actual frames

This cell just prints the real folder structure so I can check it before Step 3 parses it.


In [ ]:
print("Top-level contents of the download:")
for p in sorted(SOCCERNET_DIR.iterdir()):
    print(" ", p.relative_to(SOCCERNET_DIR))

gt_files = list(SOCCERNET_DIR.rglob("gt/gt.txt"))
print(f"\nFound {len(gt_files)} gt.txt files (one per sequence)")

if gt_files:
    seq_dir = gt_files[0].parent.parent
    print(f"\nSample sequence: {seq_dir.relative_to(SOCCERNET_DIR)}")
    print("Contents:")
    for f in sorted(seq_dir.rglob("*")):
        if f.is_file():
            print(" ", f.relative_to(seq_dir))

    print("\nFirst 5 lines of gt.txt:")
    for line in (seq_dir / "gt" / "gt.txt").read_text().splitlines()[:5]:
        print(" ", line)

    seqinfo_path = seq_dir / "seqinfo.ini"
    if seqinfo_path.exists():
        print("\nseqinfo.ini:")
        print(seqinfo_path.read_text())

    gameinfo_path = seq_dir / "gameinfo.ini"
    if gameinfo_path.exists():
        print("gameinfo.ini (first 20 lines):")
        print("\n".join(gameinfo_path.read_text().splitlines()[:20]))
    else:
        print("\n[!] No gameinfo.ini found next to gt.txt — Step 3's role lookup will need "
              "a different source file. Check the sequence contents printed above for the "
              "actual metadata filename and update load_track_roles() accordingly.")
else:
    print("\n[!] No gt.txt files found anywhere under SOCCERNET_DIR — check Step 1's download "
          "actually completed (print SOCCERNET_DIR's contents recursively to debug).")


## Step 3 — Convert to YOLO format

For each sequence:
1. Read `gameinfo.ini` to map each `track_id` to a role, collapsing SoccerNet's finer
   categories (`player team left`, `player team right`, etc.) down into my 4 classes.
2. Read `gt.txt` for the boxes, keeping every `FRAME_STRIDE`-th frame.
3. Convert boxes from MOT pixel format to YOLO's normalized `(class_id, cx, cy, w, h)`.
4. Save the image + label file into `pitchvision_dataset/{images,labels}/{train,val}/`.

If Step 2 showed a different file or key format than what's assumed here, `load_track_roles()`
and `classify_role()` below are the two functions to fix.


In [ ]:
ROLE_KEYWORDS = [
    ("ball", "ball"),
    ("goalkeeper", "goalkeeper"),
    ("keeper", "goalkeeper"),
    ("referee", "referee"),
    ("player", "player"),
]

def classify_role(role_text: str):
    """Turn a role string like 'player; team left' into one of CLASSES, or None."""
    text = role_text.lower()
    for needle, cls in ROLE_KEYWORDS:
        if needle in text:
            return cls
    return None  # not recognized (e.g. "staff"), drop this track


def load_track_roles(gameinfo_path: Path) -> dict:
    """Read gameinfo.ini into a {track_id: class_name} dict."""
    if not gameinfo_path.exists():
        return {}
    cfg = configparser.ConfigParser()
    cfg.read(gameinfo_path)
    roles = {}
    for section in cfg.sections():
        for key, value in cfg.items(section):
            m = re.match(r"trackletid_(\d+)", key)  # configparser lowercases keys
            if m:
                cls = classify_role(value)
                if cls:
                    roles[int(m.group(1))] = cls
    return roles


def find_sequence_dirs(root: Path):
    """Find sequence folders: anything with both gt/gt.txt and seqinfo.ini."""
    seqs = []
    for gt_file in root.rglob("gt/gt.txt"):
        seq_dir = gt_file.parent.parent
        if (seq_dir / "seqinfo.ini").exists():
            seqs.append(seq_dir)
    return sorted(set(seqs))


def read_seqinfo(seqinfo_path: Path) -> dict:
    cfg = configparser.ConfigParser()
    cfg.read(seqinfo_path)
    sec = cfg["Sequence"]
    return {
        "imDir": sec.get("imDir", "img1"),
        "imExt": sec.get("imExt", ".jpg"),
        "imWidth": int(sec["imWidth"]),
        "imHeight": int(sec["imHeight"]),
    }


def convert_sequence(seq_dir: Path, out_images: Path, out_labels: Path, stride: int, prefix: str):
    info = read_seqinfo(seq_dir / "seqinfo.ini")
    roles = load_track_roles(seq_dir / "gameinfo.ini")
    if not roles:
        print(f"  [skip] {seq_dir.name}: no usable roles found in gameinfo.ini")
        return 0, {c: 0 for c in CLASSES}

    gt = pd.read_csv(
        seq_dir / "gt" / "gt.txt", header=None,
        names=["frame", "track_id", "bb_left", "bb_top", "bb_w", "bb_h", "conf", "x", "y", "z"],
    )
    img_dir = seq_dir / info["imDir"]
    w, h = info["imWidth"], info["imHeight"]

    written = 0
    class_counts = {c: 0 for c in CLASSES}
    for frame_id, group in gt.groupby("frame"):
        if int(frame_id) % stride != 0:
            continue
        img_path = img_dir / f"{int(frame_id):06d}{info['imExt']}"
        if not img_path.exists():
            continue

        lines = []
        for row in group.itertuples(index=False):
            cls = roles.get(int(row.track_id))
            if cls is None:
                continue  # unrecognized track, skip
            cx = (row.bb_left + row.bb_w / 2) / w
            cy = (row.bb_top + row.bb_h / 2) / h
            bw = row.bb_w / w
            bh = row.bb_h / h
            cx, cy, bw, bh = (min(max(v, 0.0), 1.0) for v in (cx, cy, bw, bh))
            lines.append(f"{CLASS_TO_ID[cls]} {cx:.6f} {cy:.6f} {bw:.6f} {bh:.6f}")
            class_counts[cls] += 1

        if not lines:
            continue

        out_name = f"{prefix}_{int(frame_id):06d}"
        shutil.copyfile(img_path, out_images / f"{out_name}.jpg")
        (out_labels / f"{out_name}.txt").write_text("\n".join(lines))
        written += 1

    return written, class_counts


In [ ]:
seq_dirs = find_sequence_dirs(SOCCERNET_DIR)
print(f"Found {len(seq_dirs)} labeled sequences")
assert seq_dirs, "No sequences found — check Step 2's output before continuing."

random.shuffle(seq_dirs)
n_val = max(1, int(len(seq_dirs) * VAL_FRACTION))
val_seqs = set(seq_dirs[:n_val])
train_seqs = set(seq_dirs[n_val:])
print(f"{len(train_seqs)} sequences -> train, {len(val_seqs)} sequences -> val "
      f"(split by whole sequence, not by frame, so near-duplicate frames from the same "
      f"clip never leak across the split)")

for split in ["train", "val"]:
    (YOLO_DATASET_DIR / "images" / split).mkdir(parents=True, exist_ok=True)
    (YOLO_DATASET_DIR / "labels" / split).mkdir(parents=True, exist_ok=True)

total_counts = {c: 0 for c in CLASSES}
total_frames = 0
for seq_dir in tqdm(seq_dirs, desc="Converting sequences"):
    split = "val" if seq_dir in val_seqs else "train"
    n, counts = convert_sequence(
        seq_dir,
        YOLO_DATASET_DIR / "images" / split,
        YOLO_DATASET_DIR / "labels" / split,
        stride=FRAME_STRIDE,
        prefix=seq_dir.name,
    )
    total_frames += n
    for c in CLASSES:
        total_counts[c] += counts.get(c, 0)

print(f"\nConverted {total_frames} frames total")
print("Instance counts per class:", total_counts)
if any(v == 0 for v in total_counts.values()):
    print("\n[!] At least one class has zero instances — re-check ROLE_KEYWORDS in Step 3 "
          "against the real role strings printed in Step 2's gameinfo.ini sample.")


## Step 4 — Write the dataset YAML

In [ ]:
DATA_YAML = YOLO_DATASET_DIR / "data.yaml"

data_yaml = {
    "path": str(YOLO_DATASET_DIR),
    "train": "images/train",
    "val": "images/val",
    "names": {i: c for i, c in enumerate(CLASSES)},
}
with open(DATA_YAML, "w") as f:
    yaml.safe_dump(data_yaml, f, sort_keys=False)

print(DATA_YAML.read_text())


## Step 5 — Check a converted sample

Quick visual check before training. If the boxes don't line up with real players/the ball,
go fix the parsing in Step 3 first, don't waste GPU hours training on bad labels.


In [ ]:
sample_labels = sorted((YOLO_DATASET_DIR / "labels" / "train").glob("*.txt"))
assert sample_labels, "No converted labels found — Step 3/4 didn't produce any training data."

sample_label = random.choice(sample_labels)
sample_image = YOLO_DATASET_DIR / "images" / "train" / f"{sample_label.stem}.jpg"

img = cv2.cvtColor(cv2.imread(str(sample_image)), cv2.COLOR_BGR2RGB)
h, w = img.shape[:2]

fig, ax = plt.subplots(figsize=(10, 6))
ax.imshow(img)
for line in sample_label.read_text().splitlines():
    cls_id, cx, cy, bw, bh = map(float, line.split())
    cls_id = int(cls_id)
    x1 = (cx - bw / 2) * w
    y1 = (cy - bh / 2) * h
    ax.add_patch(plt.Rectangle((x1, y1), bw * w, bh * h, fill=False, edgecolor="lime", linewidth=2))
    ax.text(x1, max(y1 - 4, 0), CLASSES[cls_id], color="lime", fontsize=10, weight="bold")
ax.set_title(f"Sanity check: {sample_label.stem}")
ax.axis("off")
plt.show()


## Step 6 — Train

Fine-tunes the pretrained `yolo26n.pt` checkpoint on the converted dataset.


In [ ]:
model = YOLO(BASE_MODEL)

train_results = model.train(
    data=str(DATA_YAML),
    epochs=EPOCHS,
    imgsz=IMGSZ,
    batch=BATCH,
    patience=PATIENCE,
    device=DEVICE,
    project=str(RUNS_DIR),
    name=RUN_NAME,
    plots=True,
)

RUN_DIR = RUNS_DIR / RUN_NAME
print("Training run saved to:", RUN_DIR)


## Step 7 — Validate

In [ ]:
best_weights = RUN_DIR / "weights" / "best.pt"
best_model = YOLO(str(best_weights))

metrics = best_model.val(data=str(DATA_YAML), plots=True, project=str(RUNS_DIR), name=f"{RUN_NAME}_val")
print("mAP50:    ", metrics.box.map50)
print("mAP50-95: ", metrics.box.map)
print("Precision:", metrics.box.mp)
print("Recall:   ", metrics.box.mr)

from IPython.display import Image, display
cm_path = RUNS_DIR / f"{RUN_NAME}_val" / "confusion_matrix.png"
if cm_path.exists():
    display(Image(filename=str(cm_path)))


## Step 8 — Look at some predictions

In [ ]:
val_images = list((YOLO_DATASET_DIR / "images" / "val").glob("*.jpg"))
sample_paths = random.sample(val_images, min(6, len(val_images)))

fig, axes = plt.subplots(2, 3, figsize=(18, 8))
for ax, img_path in zip(axes.flatten(), sample_paths):
    result = best_model.predict(str(img_path), conf=0.25, imgsz=IMGSZ, verbose=False)[0]
    annotated = result.plot()[..., ::-1]  # BGR -> RGB
    ax.imshow(annotated)
    ax.set_title(img_path.name, fontsize=8)
    ax.axis("off")
plt.tight_layout()
plt.show()


## Step 9 — Export the files I need

Copies the weights and training log into one folder so they're easy to find in Kaggle's Output tab.


In [ ]:
FINAL_DIR = WORK_DIR / "final_export"
FINAL_DIR.mkdir(exist_ok=True)

shutil.copy(RUN_DIR / "weights" / "best.pt", FINAL_DIR / "best.pt")
shutil.copy(RUN_DIR / "results.csv", FINAL_DIR / "results.csv")

results_png = RUN_DIR / "results.png"
if results_png.exists():
    shutil.copy(results_png, FINAL_DIR / "results.png")

print("Exported to:", FINAL_DIR)
for f in sorted(FINAL_DIR.iterdir()):
    print(f"  {f.name:<16} {f.stat().st_size / 1e6:6.1f} MB")


## What to download

Open Kaggle's **Output** tab and grab these from `final_export/`:

| File | Goes to |
|---|---|
| `best.pt` | `models/football_best.pt` (rename it, replace the old file) |
| `results.csv` | `assets/results.csv` (replace the old file) |
| `results.png` *(optional)* | `assets/results.png`, only used in the README |

The rename matters: the app looks for `models/football_best.pt` first, so if you leave
it named `best.pt` it'll just keep loading the old model.

Replacing `results.csv` also updates the Accuracy page with this model's real numbers.
Nothing else needs to change, just restart `python server.py` after.
